In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier


In [2]:
from imblearn.over_sampling import SMOTE


In [3]:
data=pd.read_csv('/content/credit crad fraud detection.csv.zip',compression='zip')
print(data.head())

   TransactionID             TransactionDate   Amount  MerchantID  \
0              1  2024-04-03 14:15:35.462794  4189.27         688   
1              2  2024-03-19 13:20:35.462824  2659.71         109   
2              3  2024-01-08 10:08:35.462834   784.00         394   
3              4  2024-04-13 23:50:35.462850  3514.40         944   
4              5  2024-07-12 18:51:35.462858   369.07         475   

  TransactionType      Location  IsFraud  
0          refund   San Antonio        0  
1          refund        Dallas        0  
2        purchase      New York        0  
3        purchase  Philadelphia        0  
4        purchase       Phoenix        0  


In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   TransactionID    100000 non-null  int64  
 1   TransactionDate  100000 non-null  object 
 2   Amount           100000 non-null  float64
 3   MerchantID       100000 non-null  int64  
 4   TransactionType  100000 non-null  object 
 5   Location         100000 non-null  object 
 6   IsFraud          100000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 5.3+ MB
None


In [5]:
print(data['IsFraud'].value_counts())

IsFraud
0    99000
1     1000
Name: count, dtype: int64


In [6]:
X=data.drop('IsFraud',axis=1)
y=data['IsFraud']

In [7]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
scaler=StandardScaler()


X_train_df, X_test_df, _, _ = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


non_numerical_cols = X_train_df.select_dtypes(exclude=np.number).columns


X_train_numerical = X_train_df.drop(columns=non_numerical_cols)
X_test_numerical = X_test_df.drop(columns=non_numerical_cols)


X_train = scaler.fit_transform(X_train_numerical)
X_test = scaler.transform(X_test_numerical)

In [9]:
smote=SMOTE(random_state=42)
X_train_balanced, y_train_balanced=smote.fit_resample(X_train,y_train)
print("Before Balancing:")
print(y_train.value_counts())
print("After Balancing:")
print(pd.Series(y_train_balanced).value_counts())


Before Balancing:
IsFraud
0    79200
1      800
Name: count, dtype: int64
After Balancing:
IsFraud
0    79200
1    79200
Name: count, dtype: int64


In [10]:
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)
model=model.fit(X_train_balanced,y_train_balanced)


In [11]:
y_pred=model.predict(X_test)

In [12]:
for i in range(10):
  if y_pred[i]==1:
    print("Transaction",i+1,":Fraud Transaction")
  else:
    print("Transaction",i+1,":Normal Transaction")

Transaction 1 :Normal Transaction
Transaction 2 :Normal Transaction
Transaction 3 :Normal Transaction
Transaction 4 :Normal Transaction
Transaction 5 :Normal Transaction
Transaction 6 :Normal Transaction
Transaction 7 :Normal Transaction
Transaction 8 :Normal Transaction
Transaction 9 :Normal Transaction
Transaction 10 :Normal Transaction


In [13]:
print("Number of Fraud Predictions:", np.sum(y_pred==1))
print("Number of Normal Predictions:", np.sum(y_pred==0))

Number of Fraud Predictions: 1663
Number of Normal Predictions: 18337


In [14]:
cm=confusion_matrix(y_test,y_pred)
print("confusion matrix:")
print(cm)

confusion matrix:
[[18151  1649]
 [  186    14]]


In [15]:
accuracy=accuracy_score(y_test,y_pred)
print("Model Accuracy:",accuracy*100,"%")
print("\nClassification Report:\n", classification_report(y_test,y_pred))

Model Accuracy: 90.825 %

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.92      0.95     19800
           1       0.01      0.07      0.02       200

    accuracy                           0.91     20000
   macro avg       0.50      0.49      0.48     20000
weighted avg       0.98      0.91      0.94     20000

